### PACOTES

In [4]:
import pandas as pd
import numpy as np
import time

from itertools import combinations

from scipy.stats import (
    ks_2samp,
    wasserstein_distance,
    spearmanr
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    roc_auc_score,
    mutual_info_score
)

from sklearn.preprocessing import RobustScaler


from joblib import Parallel, delayed

from IPython.display import display 
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


In [10]:
# =========================
# CARREGA O CSV ORIGINAL
# =========================

df = pd.read_csv("visu_resumo_features.csv")

# =========================
# MÉTRICAS UTILIZADAS
# =========================

metricas_positivas = [
    "AUC_PR",
    "F1_Score",
    "MCC",
    "KS"
]

metrica_negativa = "Log_Loss"

# =========================
# NORMALIZAÇÃO 0-1
# =========================

scaler = MinMaxScaler()

# Métricas onde MAIOR = MELHOR
df_norm = pd.DataFrame(
    scaler.fit_transform(df[metricas_positivas]),
    columns=metricas_positivas
)

# =========================
# LOG LOSS
# MENOR = MELHOR
# =========================

logloss_norm = scaler.fit_transform(
    df[[metrica_negativa]]
)

# Inversão do Log Loss
df_norm["Log_Loss"] = 1 - logloss_norm.flatten()

# =========================
# SCORE FINAL
# =========================

df["Score_Final"] = (
    df_norm[
        ["AUC_PR", "F1_Score", "MCC", "KS", "Log_Loss"]
    ].sum(axis=1)
) / 5

# =========================
# ORDENA DO MELHOR PARA PIOR
# =========================

df = df.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

# =========================
# POSIÇÃO DO RANKING
# =========================

df["Posicao_Rank"] = range(1, len(df) + 1)

# =========================
# SALVA NOVO CSV
# =========================

df.to_csv(
    "1x1visu_scores.csv",
    index=False
)

print("Arquivo 1x1visu_scores.csv criado com sucesso.")

Arquivo 1x1visu_scores.csv criado com sucesso.


 ### CRIANDO CSV DE COMBINACOES 2X2 

In [14]:
import time
import itertools
import numpy as np
import pandas as pd

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss
)

from scipy.stats import ks_2samp

# ==========================================
# CARREGA DADOS
# ==========================================

df = pd.read_csv("creditcard.csv")

ranking = pd.read_csv("visu_resumo_features.csv")

# ==========================================
# TARGET
# ==========================================

TARGET = "status_fraude"

# ==========================================
# FEATURES
# ==========================================

# Opcional:
# usar apenas top 30 para acelerar

features = ranking.head(30)["Feature"].tolist()

# ==========================================
# RESULTADOS
# ==========================================

resultados = []

# ==========================================
# OBJETOS REUTILIZÁVEIS
# ==========================================

scaler = StandardScaler()

# ==========================================
# LOOP DAS COMBINAÇÕES 2x2
# ==========================================

total_combinacoes = len(list(itertools.combinations(features, 2)))

contador = 0

for f1, f2 in itertools.combinations(features, 2):

    contador += 1

    try:

        # ==========================================
        # DADOS
        # ==========================================

        temp = df[[f1, f2, TARGET]].dropna()

        # Segurança
        if len(temp) == 0:
            continue

        X = temp[[f1, f2]]
        y_real = temp[TARGET]

        # Segurança
        if y_real.nunique() < 2:
            continue

        # ==========================================
        # ESCALONAMENTO
        # ==========================================

        X_scaled = scaler.fit_transform(X)

        # ==========================================
        # GMM
        # ==========================================

        inicio = time.perf_counter()

        gmm = GaussianMixture(
            n_components=2,
            covariance_type='full',
            random_state=42,
            reg_covar=1e-6,
            n_init=3
        )

        gmm.fit(X_scaled)

        fim = time.perf_counter()

        # ==========================================
        # CLUSTERS
        # ==========================================

        clusters = gmm.predict(X_scaled)

        ct = pd.crosstab(clusters, y_real)

        # Segurança
        if 1 not in ct.columns:
            continue

        cluster_fraude = ct[1].idxmax()

        # ==========================================
        # PROBABILIDADES
        # ==========================================

        probabilidades = gmm.predict_proba(
            X_scaled
        )[:, cluster_fraude]

        # Evita problemas numéricos
        probabilidades = np.clip(
            probabilidades,
            1e-15,
            1 - 1e-15
        )

        # ==========================================
        # PREDIÇÃO
        # ==========================================

        y_pred = (
            probabilidades >= 0.50
        ).astype(int)

        # ==========================================
        # AUC-PR
        # ==========================================

        precision_vals, recall_vals, _ = (
            precision_recall_curve(
                y_real,
                probabilidades
            )
        )

        auc_pr = auc(
            recall_vals,
            precision_vals
        )

        # ==========================================
        # F1
        # ==========================================

        f1_final = f1_score(
            y_real,
            y_pred
        )

        # ==========================================
        # MCC
        # ==========================================

        mcc = matthews_corrcoef(
            y_real,
            y_pred
        )

        # ==========================================
        # KS
        # ==========================================

        ks = ks_2samp(
            probabilidades[y_real == 0],
            probabilidades[y_real == 1]
        ).statistic

        # ==========================================
        # LOG LOSS
        # ==========================================

        ll = log_loss(
            y_real,
            probabilidades
        )

        # ==========================================
        # SALVA RESULTADO
        # ==========================================

        resultados.append({

            "Combinacao": f"{f1} | {f2}",

            "Feature_1": f1,
            "Feature_2": f2,

            "AUC_PR": auc_pr,
            "F1_Score": f1_final,
            "MCC": mcc,
            "KS": ks,
            "Log_Loss": ll,

            "Tempo": fim - inicio

        })

        print(
            f"[{contador}/{total_combinacoes}] "
            f"OK -> {f1} + {f2}"
        )

    except Exception as e:

        print(
            f"[{contador}/{total_combinacoes}] "
            f"ERRO -> {f1} + {f2}"
        )

        print(e)

# ==========================================
# DATAFRAME FINAL
# ==========================================

df_resultados = pd.DataFrame(resultados)

# ==========================================
# NORMALIZAÇÃO
# ==========================================

metricas_positivas = [
    "AUC_PR",
    "F1_Score",
    "MCC",
    "KS"
]

metrica_negativa = "Log_Loss"

score_scaler = MinMaxScaler()

# Métricas positivas
df_norm = pd.DataFrame(

    score_scaler.fit_transform(
        df_resultados[metricas_positivas]
    ),

    columns=metricas_positivas
)

# Log Loss invertido
logloss_norm = score_scaler.fit_transform(
    df_resultados[[metrica_negativa]]
)

df_norm["Log_Loss"] = (
    1 - logloss_norm.flatten()
)

# ==========================================
# SCORE FINAL
# ==========================================

df_resultados["Score_Final"] = (

    df_norm[
        [
            "AUC_PR",
            "F1_Score",
            "MCC",
            "KS",
            "Log_Loss"
        ]
    ].sum(axis=1)

) / 5

# ==========================================
# ORDENA
# ==========================================

df_resultados = df_resultados.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

# ==========================================
# POSIÇÃO RANK
# ==========================================

df_resultados["Posicao_Rank"] = (
    df_resultados.index + 1
)

# ==========================================
# SALVA CSV
# ==========================================

df_resultados.to_csv(
    "2x2_visu_scores.csv",
    index=False
)

print("\nArquivo 2x2_visu_scores.csv criado com sucesso.")

[1/435] OK -> V1 + V2
[2/435] OK -> V1 + V3
[3/435] OK -> V1 + V4
[4/435] OK -> V1 + V5
[5/435] OK -> V1 + V6
[6/435] OK -> V1 + V7
[7/435] OK -> V1 + V8
[8/435] OK -> V1 + V9
[9/435] OK -> V1 + V10
[10/435] OK -> V1 + V11
[11/435] OK -> V1 + V12
[12/435] OK -> V1 + V13
[13/435] OK -> V1 + V14
[14/435] OK -> V1 + V15
[15/435] OK -> V1 + V16
[16/435] OK -> V1 + V17
[17/435] OK -> V1 + V18
[18/435] OK -> V1 + V19
[19/435] OK -> V1 + V20
[20/435] OK -> V1 + V21
[21/435] OK -> V1 + V22
[22/435] OK -> V1 + V23
[23/435] OK -> V1 + V24
[24/435] OK -> V1 + V25
[25/435] OK -> V1 + V26
[26/435] OK -> V1 + V27
[27/435] OK -> V1 + V28
[28/435] OK -> V1 + tempo_desde_a_primeira_transacao
[29/435] OK -> V1 + valor_de_transacao
[30/435] OK -> V2 + V3
[31/435] OK -> V2 + V4
[32/435] OK -> V2 + V5
[33/435] OK -> V2 + V6
[34/435] OK -> V2 + V7
[35/435] OK -> V2 + V8
[36/435] OK -> V2 + V9
[37/435] OK -> V2 + V10
[38/435] OK -> V2 + V11
[39/435] OK -> V2 + V12
[40/435] OK -> V2 + V13
[41/435] OK -> V2 + 

 ### CRIANDO CSV DE COMBINACOES 3x3

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss
)

from scipy.stats import ks_2samp

from openTSNE import TSNE

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# DADOS
# ==========================================

BASE = pd.read_csv("creditcard.csv")
RANKING = pd.read_csv("3x3_visu_scores.csv")

TARGET = "status_fraude"

# ==========================================
# TOP 1 FEATURES
# ==========================================

TOP1 = (
    RANKING
    .sort_values(by="Score_Final", ascending=False)
    .head(1)
    .reset_index(drop=True)
)

row = TOP1.iloc[0]

f1, f2, f3 = row["Feature_1"], row["Feature_2"], row["Feature_3"]

print(f"\nTOP 1 -> {f1} + {f2} + {f3}")

# ==========================================
# BASE
# ==========================================

temp = BASE[[f1, f2, f3, TARGET]].dropna()

X = temp[[f1, f2, f3]]
y = temp[TARGET]

# ==========================================
# ESCALONAMENTO
# ==========================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# ==========================================
# GMM ORIGINAL
# ==========================================

gmm = GaussianMixture(
    n_components=2,
    covariance_type='full',
    random_state=42,
    reg_covar=1e-6,
    n_init=3
)

gmm.fit(X_scaled)

clusters = gmm.predict(X_scaled)

ct = pd.crosstab(clusters, y)
cluster_fraude = ct[1].idxmax()

probs = gmm.predict_proba(X_scaled)[:, cluster_fraude]
probs = np.clip(probs, 1e-15, 1 - 1e-15)

y_pred = (probs >= 0.5).astype(int)

cm_original = confusion_matrix(y, y_pred, normalize='true')

# ==========================================
# AMOSTRAGEM TSNE
# ==========================================

nao_fraude = temp[temp[TARGET] == 0]
fraude = temp[temp[TARGET] == 1]

nao_fraude_sample = nao_fraude.sample(frac=0.05, random_state=42)

temp_tsne = pd.concat([nao_fraude_sample, fraude]).reset_index(drop=True)

X_tsne = temp_tsne[[f1, f2, f3]]
y_tsne = temp_tsne[TARGET]

X_tsne_scaled = scaler.fit_transform(X_tsne).astype(np.float32)

# ==========================================
# t-SNE 3D (VERSÃO COMPATÍVEL)
# ==========================================

print("\nRodando t-SNE 3D (openTSNE modo estável)...\n")

tsne = TSNE(
    n_components=3,
    perplexity=30,
    metric="euclidean",
    initialization="pca",
    random_state=42,
    n_jobs=7,

    # 🔥 ESSENCIAL: evita interpolação incompatível
    negative_gradient_method="bh",

    verbose=True
)

coords = tsne.fit(X_tsne_scaled)
coords = np.asarray(coords)

# ==========================================
# GMM NO ESPAÇO TSNE
# ==========================================

gmm_tsne = GaussianMixture(
    n_components=2,
    covariance_type='full',
    random_state=42,
    reg_covar=1e-6,
    n_init=3
)

gmm_tsne.fit(coords)

clusters_tsne = gmm_tsne.predict(coords)

ct_tsne = pd.crosstab(clusters_tsne, y_tsne)
cluster_fraude_tsne = ct_tsne[1].idxmax()

probs_tsne = gmm_tsne.predict_proba(coords)[:, cluster_fraude_tsne]
probs_tsne = np.clip(probs_tsne, 1e-15, 1 - 1e-15)

y_pred_tsne = (probs_tsne >= 0.5).astype(int)

# ==========================================
# MÉTRICAS
# ==========================================

precision_vals, recall_vals, _ = precision_recall_curve(y_tsne, probs_tsne)
auc_pr_tsne = auc(recall_vals, precision_vals)

f1_tsne = f1_score(y_tsne, y_pred_tsne)
mcc_tsne = matthews_corrcoef(y_tsne, y_pred_tsne)

ks_tsne = ks_2samp(
    probs_tsne[y_tsne == 0],
    probs_tsne[y_tsne == 1]
).statistic

logloss_tsne = log_loss(y_tsne, probs_tsne)

# ==========================================
# VISUALIZAÇÃO
# ==========================================

df_original = pd.DataFrame({
    "x": X[f1],
    "y": X[f2],
    "z": X[f3],
    "classe": y.astype(str)
})

scatter_original = px.scatter_3d(
    df_original,
    x="x", y="y", z="z",
    color="classe",
    opacity=0.45
)

df_tsne = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "z": coords[:, 2],
    "classe": y_tsne.astype(str)
})

scatter_tsne = px.scatter_3d(
    df_tsne,
    x="x", y="y", z="z",
    color="classe",
    opacity=0.6
)

# ==========================================
# MATRIZES
# ==========================================

heat_original = go.Heatmap(
    z=cm_original,
    colorscale="Reds",
    showscale=False
)

cm_tsne = confusion_matrix(y_tsne, y_pred_tsne, normalize='true')

heat_tsne = go.Heatmap(
    z=cm_tsne,
    colorscale="Reds",
    showscale=False
)

# ==========================================
# FIGURA FINAL
# ==========================================

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "scene"}, {"type": "heatmap"}],
        [{"type": "scene"}, {"type": "heatmap"}]
    ],
    subplot_titles=[
        "Original 3D",
        "Matriz Original",
        "t-SNE 3D",
        "Matriz t-SNE"
    ]
)

for trace in scatter_original.data:
    fig.add_trace(trace, row=1, col=1)

fig.add_trace(heat_original, row=1, col=2)

for trace in scatter_tsne.data:
    fig.add_trace(trace, row=2, col=1)

fig.add_trace(heat_tsne, row=2, col=2)

# ==========================================
# LAYOUT
# ==========================================

fig.update_layout(
    height=1500,
    width=1800,
    template="plotly_white",
    title=f"TOP 1 -> {f1} + {f2} + {f3}",
    showlegend=False
)

# ==========================================
# EXPORT
# ==========================================

fig.write_html("top1_3x3_tsne_gmm.html")

print("\nArquivo top1_3x3_tsne_gmm.html criado com sucesso.")


Total de combinações: 4060



[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:   43.8s
[Parallel(n_jobs=7)]: Done  11 tasks      | elapsed:  1.8min
[Parallel(n_jobs=7)]: Done  18 tasks      | elapsed:  2.4min
[Parallel(n_jobs=7)]: Done  27 tasks      | elapsed:  3.4min
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:  3.9min
[Parallel(n_jobs=7)]: Done  47 tasks      | elapsed:  4.9min
[Parallel(n_jobs=7)]: Done  58 tasks      | elapsed:  5.9min
[Parallel(n_jobs=7)]: Done  71 tasks      | elapsed:  6.9min
[Parallel(n_jobs=7)]: Done  84 tasks      | elapsed:  8.0min
[Parallel(n_jobs=7)]: Done  99 tasks      | elapsed:  9.0min
[Parallel(n_jobs=7)]: Done 114 tasks      | elapsed: 10.4min
[Parallel(n_jobs=7)]: Done 131 tasks      | elapsed: 12.1min
[Parallel(n_jobs=7)]: Done 148 tasks      | elapsed: 13.7min
[Parallel(n_jobs=7)]: Done 167 tasks      | elapsed: 15.2min
[Parallel(n_jobs=7)]: Done 186 tasks      | elapsed: 16.8min
[Parallel(


Arquivo 3x3_visu_scores.csv criado com sucesso.

Tempo total: 17359.20 segundos
